# 02 Process Full Data
**Goal:** Turn the two CSV files into a single parquet file

## Setup

In [1]:
# Imports
from pathlib import Path

import pandas as pd

from nyc_congestion_bus_speeds.cleaning import snake_case_columns

In [2]:
# Folders
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Data paths
CSV_2023_24 = RAW_DIR / "segment_speeds_2023_24.csv"
CSV_2025 = RAW_DIR / "segment_speeds_2025.csv"

OUT_PATH = PROCESSED_DIR / "segment_speeds_peak.parquet"

# Settings
END_YEAR = 2026
END_MONTH = 8

# Peak is from 5am-9pm.
PEAK_START_HOUR = 5
PEAK_END_HOUR = 20

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

USECOLS = [
    "Year", "Month", "Day of Week", "Hour of Day",
    "Route ID", "Direction", "Borough", "Route Type",
    "Timepoint Stop ID", "Next Timepoint Stop ID",
    "Timepoint Stop Latitude", "Timepoint Stop Longitude",
    "Next Timepoint Stop Latitude", "Next Timepoint Stop Longitude",
    "Road Distance", "Average Travel Time", "Bus Trip Count",
]



## 1. Load both files
**Goal:** Load both CSVs

In [3]:
raw24 = snake_case_columns(pd.read_csv(CSV_2023_24, usecols=USECOLS, engine="pyarrow"))
raw24.shape

(11656097, 17)

In [4]:
raw25 = snake_case_columns(pd.read_csv(CSV_2025, usecols=USECOLS, engine="pyarrow"))
raw25.shape

(9694073, 17)

In [5]:
# Compare column types to make sure they match
pd.concat([raw24.dtypes, raw25.dtypes], axis=1, keys=["raw24", "raw25"])

,raw24,raw25
year,int64,int64
month,int64,int64
day_of_week,str,str
hour_of_day,int64,int64
route_id,str,str
direction,str,str
borough,str,str
route_type,str,str
timepoint_stop_id,int64,int64
next_timepoint_stop_id,int64,int64


In [6]:
speeds = pd.concat([raw24, raw25], ignore_index=True)
del raw24, raw25
speeds.shape

(21350170, 17)

**Findings:**
- Column types are identical.

## 2. Filter to the study period and peak hours
**Goal:** Filter data to the end of August 2026, keeping weekday peak hours only and record how much is dropped.

In [7]:
n_start = len(speeds)

in_weekday = speeds["day_of_week"].isin(WEEKDAYS)
in_peak_hours = speeds["hour_of_day"].between(PEAK_START_HOUR, PEAK_END_HOUR)
in_study_period = (
    (speeds["year"] < END_YEAR)
    | ((speeds["year"] == END_YEAR) & (speeds["month"] <= END_MONTH))
)
speeds = speeds[ in_weekday & in_peak_hours & in_study_period ]

print(f"Kept {len(speeds):,} of {n_start:,} rows ({len(speeds) / n_start:.1%})")

Kept 11,991,496 of 21,350,170 rows (56.2%)


**Findings:**
- Kept 11.99M of 21.35M rows (56.2%). This is more than the roughly 48% expected from hours alone (5/7 days * 16/24 hours), likely due to overnight and weekend hours having less service, and thus fewer rows.

## 3. Check what's left
**Goal:** Confirm all 44 expected months are present, and look for possible erroneous values.

In [8]:
display(speeds.groupby(["year", "month"]).size().unstack("month"))
display(speeds.groupby(["year", "month"])["bus_trip_count"].sum().unstack("month"))

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2023,276319.0,270841.0,268841.0,268187.0,271353.0,268551.0,265715.0,296694.0,273346.0,269620.0,272745.0,274752.0
2024,282578.0,271229.0,268647.0,268896.0,271611.0,288831.0,266493.0,287270.0,271628.0,269060.0,271744.0,273555.0
2025,280425.0,273714.0,272893.0,267789.0,272123.0,285771.0,262632.0,259376.0,268240.0,263127.0,266790.0,269372.0
2026,285320.0,281466.0,260970.0,294757.0,264401.0,279122.0,258834.0,255868.0,NaN,NaN,NaN,NaN


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2023,4846041.0,4413555.0,5177586.0,4487992.0,5033745.0,4895249.0,4527629.0,5038175.0,4444166.0,4845335.0,4721029.0,4397172.0
2024,4987026.0,4560370.0,4664184.0,4873430.0,4931120.0,4331148.0,4855248.0,4648950.0,4455523.0,5100080.0,4523772.0,4623304.0
2025,5086587.0,4243638.0,4486058.0,4701105.0,4541540.0,4645949.0,4956499.0,4624457.0,4501751.0,4811558.0,4374071.0,4875315.0
2026,4531787.0,3963178.0,4538667.0,4595963.0,4268215.0,4511253.0,4743813.0,4428211.0,NaN,NaN,NaN,NaN


In [9]:
speeds.describe().T

,count,mean,std,min,25%,50%,75%,max
year,11991496.0,2024.360819,1.068125,2023.000000,2023.000000,2024.000000,2025.000000,2026.000000
month,11991496.0,6.113115,3.367145,1.000000,3.000000,6.000000,9.000000,12.000000
hour_of_day,11991496.0,12.616565,4.562994,5.000000,9.000000,13.000000,17.000000,20.000000
timepoint_stop_id,11991496.0,386315.774654,202374.127248,100014.000000,300040.000000,401079.000000,503324.000000,992083.000000
next_timepoint_stop_id,11991496.0,431623.712838,244587.946431,100014.000000,300474.000000,402653.000000,550122.000000,992080.000000
timepoint_stop_latitude,11986921.0,40.720778,0.084814,40.502981,40.656280,40.713603,40.773561,40.932045
timepoint_stop_longitude,11986921.0,-73.924941,0.102919,-74.251470,-73.981034,-73.925580,-73.849797,-73.701373
next_timepoint_stop_latitude,11986742.0,40.720760,0.084775,40.502981,40.656280,40.713383,40.773505,40.933637
next_timepoint_stop_longitude,11986742.0,-73.925080,0.102953,-74.251470,-73.981026,-73.925681,-73.849860,-73.701230
road_distance,11991496.0,1.447548,1.652983,0.000000,0.723000,1.095000,1.590000,26.126000


In [10]:
(speeds["road_distance"] == 0).sum()

np.int64(11)

**Findings:**
- All 44 months are present, and no extra months are present. Although August 2026 is a down month relative to July 2026, the drop is consistent with previous years and not out of range.
- The minimum for `bus_trip_count` is 1, meaning hours without service have no rows. This explains the previous finding that peak-hour weekdays are overrepresented in data (56.2% versus the expected 48%).
- All coordinates are within NYC's bounds. However, about 4,600 rows of both timepoint coordinates and next timepoint coordinates are missing. These will need to be dropped or filled in from the same stop's other rows.
- There are 11 rows with a `road_distance` of 0, and `average_travel_time` goes as low as about 0.3 seconds. Travel times are dependent on distance, so they don't inherently prove an error. Therefore, in the future we will clean up based on implied speed (distance / time) instead.

## 4. Save
**Goal:** Save the filtered data for future notebooks.

In [11]:
speeds.to_parquet(OUT_PATH, index=False)
print(f"Saved {OUT_PATH.stat().st_size / 1e6:.0f} MB")

Saved 218 MB


In [12]:
print(f"Saved data matches: {pd.read_parquet(OUT_PATH).shape == speeds.shape}")

Saved data matches: True


## Summary
The saved file contains 11.99M rows from January 2023 through August 2026 covering weekdays during peak hours. All months are present and complete, and the types are consistent between files. Zero-value distances, implausibly short travel times, and ~4,600 rows with missing coordinates still need cleaning, which is saved for the future.